# NNUE v4: balanced pilot

Fresh training using the existing dataset. This is an experiment, not a stronger agent yet. Do not run the old notebook's training cells. Preserve v3. A GPU is useful for training. Raw sparse-position errors must improve before deployment; games remain mandatory.

In [ ]:
from pathlib import Path
import subprocess, sys, os
from google.colab import drive
drive.mount('/content/drive')
repo = Path('/content/Einsteinanium-balanced')
if not repo.exists():
    subprocess.run(['git','clone','--branch','codex/search-v2','https://github.com/divitkashyap/aichessathon-starter.git',str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-q','python-chess','numpy','numba','torch'], check=True)
assert Path('tools/train_nnue_balanced.py').is_file(), 'Code missing: obtain current branch before continuing'
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-p','test_balanced_training.py'], check=True)

In [ ]:
from datetime import datetime, timezone
DATASET = '/content/drive/MyDrive/einsteinanium/nnue-v1/data'
RUN = Path('/content/drive/MyDrive/einsteinanium') / ('nnue-v4-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
assert Path(DATASET).is_dir(), 'Existing dataset missing; do not regenerate blindly'
assert not RUN.exists()
print('NEW run:', RUN)
subprocess.run([sys.executable,'-m','tools.train_nnue_balanced','--dataset',DATASET,'--run',str(RUN),'--epochs','12'], check=True)

In [ ]:
import json
reports = [(json.loads(p.read_text()), p) for p in RUN.glob('epoch-*.json')]
assert reports, 'Training reports missing'
best, report_path = min(reports, key=lambda x: x[0]['selection_score'])
checkpoint = report_path.with_suffix('.pt')
assert checkpoint.is_file()
WEIGHTS = RUN / 'einsteinanium-nnue-v4.npz'
assert not WEIGHTS.exists(), 'Export exists: preserve it'
print('Selected:', checkpoint)
subprocess.run([sys.executable,'-m','tools.export_nnue',str(checkpoint),str(WEIGHTS)], check=True)
subprocess.run([sys.executable,'-m','tools.validate_nnue','--dataset',DATASET,'--weights',str(WEIGHTS)], check=True)
print('Share weights and epoch JSON reports:', RUN)

## What to send back

Share the final validation output, the epoch JSON reports, and exported `.npz`. Keep `.pt` checkpoints in Drive. If export parity fails, stop and send the error; do not weaken the tolerance.

This changes target capping and phase weighting together: it tests a combined repair, not which ingredient caused improvement. The existing validation split is reused and not an independent playing-strength test. Lower extreme raw accuracy may be expected with capped targets; ordinary and sparse raw errors must be checked. No upload is performed.